<a href="https://colab.research.google.com/github/lmoss/onesharp/blob/main/issues/PCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title #Modified PCP Solver
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
body {
    font-family: Arial, sans-serif;
    text-align: center;
    background: #f4f6f8;
}

#tiles {
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 15px;
    max-width: 900px;
    margin: 25px auto;
}

#stack {
    display: flex;
    justify-content: center;
    flex-wrap: wrap;
    gap: 10px;
    margin-top: 20px;
}

.tile {
    border: 2px solid #333;
    border-radius: 10px;
    padding: 10px;
    width: 90px;
    cursor: pointer;
    background: white;
}

.greenTile {
    background: #c8f7c5;
    border: 3px solid green;
    font-weight: bold;
}

.match { color: green; }
.mismatch { color: red; }
.error { color: red; font-weight: bold; }

button, select {
    padding: 8px 15px;
    margin: 8px;
    font-size: 14px;
    border-radius: 6px;
    border: none;
}

button {
    background: #1976d2;
    color: white;
    cursor: pointer;
}

button:hover {
    background: #0d47a1;
}

#stats {
    margin-top: 15px;
    font-size: 14px;
}
</style>
</head>
<body>

<h2>Modified PCP: Searching for a Match</h2>

<div>
<button onclick="generateInstance()">Generate</button>
<button onclick="resetGame()">Clear Attempt</button>
<button onclick="solvePCP()">Solve</button>

<select id="searchMode">
    <option value="DFS" selected>DFS</option>
    <option value="BFS">BFS</option>
</select>

<select id="depthSelect">
    <option value="10">Depth 10</option>
    <option value="20" selected>Depth 20</option>
    <option value="30">Depth 30</option>
</select>

<select id="nodeLimitSelect">
    <option value="25000">25,000 nodes</option>
    <option value="50000" selected>50,000 nodes</option>
    <option value="100000">100,000 nodes</option>
    <option value="200000">200,000 nodes</option>
</select>
</div>

<div id="tiles"></div>

<h3 style="margin-top:40px;">Try finding a match</h3>
<div style="height:25px;"></div>

<div id="stack"></div>

<p>Top: <span id="topString"></span></p>
<p>Bottom: <span id="bottomString"></span></p>

<div id="message"></div>
<div id="stats"></div>

<script>
let tiles = [];
let topString = "";
let bottomString = "";
let stack = [];
let solverRunning = false;

const alphabet = ["a","b"];
const CHUNK_SIZE = 500;

/* ---------- Utility ---------- */

function randomString(minLen=1){
    let length = Math.floor(Math.random()*3)+minLen;
    let s="";
    for(let i=0;i<length;i++)
        s += alphabet[Math.floor(Math.random()*2)];
    return s;
}

function isProperPrefix(a,b){
    return b.startsWith(a) && a.length < b.length;
}

/* ---------- Generator ---------- */

function generateInstance(){
    tiles=[];

    let firstSymbol = alphabet[Math.floor(Math.random()*2)];
    let topFirst, bottomFirst;

    do {
        topFirst = firstSymbol + randomString(0);
        bottomFirst = firstSymbol + randomString(0);
    } while (
        !(isProperPrefix(topFirst,bottomFirst) ||
          isProperPrefix(bottomFirst,topFirst))
    );

    tiles.push({top: topFirst, bottom: bottomFirst});

    let count = Math.floor(Math.random()*3)+8;

    for(let i=1;i<count;i++){
        tiles.push({
            top: randomString(),
            bottom: randomString()
        });
    }

    resetGame();
    renderTiles();
}

/* ---------- Rendering ---------- */

function renderTiles(){
    const tilesDiv=document.getElementById("tiles");
    tilesDiv.innerHTML="";
    tiles.forEach((tile,index)=>{
        const div=document.createElement("div");
        div.className="tile";
        if(index===0) div.classList.add("greenTile");
        div.innerHTML=`<div>${tile.top}</div><hr><div>${tile.bottom}</div>`;
        div.onclick=()=>addTile(index);
        tilesDiv.appendChild(div);
    });
}

function renderStack(){
    const stackDiv=document.getElementById("stack");
    stackDiv.innerHTML="";
    stack.forEach(i=>{
        const div=document.createElement("div");
        div.className="tile";
        if(i===0) div.classList.add("greenTile");
        div.innerHTML=`<div>${tiles[i].top}</div><hr><div>${tiles[i].bottom}</div>`;
        stackDiv.appendChild(div);
    });
}

function updateStrings(){
    let t=document.getElementById("topString");
    let b=document.getElementById("bottomString");

    t.textContent=topString;
    b.textContent=bottomString;

    if(topString===bottomString && topString.length>0){
        t.className="match";
        b.className="match";
        document.getElementById("message").innerHTML=
        "<h3 class='match'>🎉 Match Found!</h3>";
    } else {
        t.className="mismatch";
        b.className="mismatch";
    }
}

function addTile(index){
    if(stack.length===0 && index!==0){
        document.getElementById("message").innerHTML =
        "<p class='error'>Must start with green tile.</p>";
        return;
    }

    stack.push(index);
    topString+=tiles[index].top;
    bottomString+=tiles[index].bottom;

    renderStack();
    updateStrings();
}

function resetGame(){
    solverRunning=false;
    topString="";
    bottomString="";
    stack=[];
    document.getElementById("stack").innerHTML="";
    document.getElementById("message").innerHTML="";
    document.getElementById("stats").innerHTML="";
    updateStrings();
}

/* ---------- Unified Solver (BFS / DFS Toggle) ---------- */

function solvePCP(){
    if(solverRunning) return;
    solverRunning=true;

    const maxDepth=parseInt(document.getElementById("depthSelect").value);
    const NODE_LIMIT=parseInt(document.getElementById("nodeLimitSelect").value);
    const mode=document.getElementById("searchMode").value;

    let frontier=[{
        seq:[0],
        top:tiles[0].top,
        bottom:tiles[0].bottom
    }];

    let visited=new Set();
    let nodesExplored=0;

    function searchChunk(){

        let count=0;

        while(frontier.length>0 && count<CHUNK_SIZE){

            let current;

            if(mode==="DFS"){
                current = frontier.pop();
            } else {
                current = frontier.shift();
            }

            nodesExplored++;

            if(nodesExplored>=NODE_LIMIT){
                solverRunning=false;
                document.getElementById("stats").innerHTML =
                mode+" stopped at node limit ("+NODE_LIMIT+")";
                return;
            }

            if(current.top===current.bottom && current.seq.length>0){
                solverRunning=false;
                animateSolution(current.seq);
                return;
            }

            if(current.seq.length<maxDepth){
                for(let i=tiles.length-1;i>=0;i--){
                    let newTop=current.top+tiles[i].top;
                    let newBottom=current.bottom+tiles[i].bottom;

                    if(!newTop.startsWith(newBottom) &&
                       !newBottom.startsWith(newTop)) continue;

                    let key=newTop+"|"+newBottom;
                    if(visited.has(key)) continue;
                    visited.add(key);

                    frontier.push({
                        seq:[...current.seq,i],
                        top:newTop,
                        bottom:newBottom
                    });
                }
            }

            count++;
        }

        document.getElementById("stats").innerHTML =
        mode+" nodes explored: "+nodesExplored;

        if(frontier.length>0 && solverRunning){
            setTimeout(searchChunk,0);
        } else {
            solverRunning=false;
            if(frontier.length===0)
                document.getElementById("stats").innerHTML +=
                " — No solution found.";
        }
    }

    searchChunk();
}

function animateSolution(sequence){
    resetGame();
    let i=0;
    function step(){
        if(i>=sequence.length) return;
        addTile(sequence[i]);
        i++;
        setTimeout(step,300);
    }
    step();
}

generateInstance();
</script>

</body>
</html>
"""

HTML(html_code)